# 02 · Autograd: graphs, custom functions and accumulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/02_autograd.ipynb) &nbsp; 📖 Concept deck: [Autograd](https://nilesh-salpe.github.io/visual-ai/pytorch/02-autograd/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** how `requires_grad`, `.grad`, `no_grad` and `detach` work, a custom `torch.autograd.Function` checked with `gradcheck`, and why gradient accumulation over micro-batches equals one big batch.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

## Task 1: A custom autograd Function

Implement SiLU, y = x·σ(x), as a `torch.autograd.Function` with `forward(ctx, x)` and `backward(ctx, grad_out)`. Save what backward needs with `ctx.save_for_backward`. Derivative: σ(x)·(1 + x·(1 − σ(x))).
The test uses `torch.autograd.gradcheck` (float64 finite differences).

In [ ]:
class SiLUFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # TODO
        raise NotImplementedError
    @staticmethod
    def backward(ctx, grad_out):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_SiLUFn(cls):
    x = torch.randn(10, dtype=torch.float64, requires_grad=True)
    assert torch.allclose(cls.apply(x), F.silu(x))
    assert torch.autograd.gradcheck(cls.apply, (x,)), "backward disagrees with finite differences"

check("SiLUFn", _test_SiLUFn, SiLUFn)

## Task 2: Gradient accumulation

Given a model, a loss function and a batch split into `k` micro-batches, compute gradients by running backward on each micro-batch loss **divided by k** without zeroing in between. Return a dict `name -> grad` (cloned). It must equal the gradient of the mean loss over the full batch.

In [ ]:
def accumulate(model, X, Y, k):
    model.zero_grad()
    # TODO: for each of k chunks: loss = F.mse_loss(model(xc), yc) / k; loss.backward()
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_accumulate(f):
    torch.manual_seed(1); m = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 1))
    X, Y = torch.randn(32, 4), torch.randn(32, 1)
    g = f(m, X, Y, 4)
    m.zero_grad(); F.mse_loss(m(X), Y).backward()
    for n, p in m.named_parameters():
        assert torch.allclose(g[n], p.grad, atol=1e-6), f"{n}: accumulated gradient differs from the full-batch gradient"

check("accumulate", _test_accumulate, accumulate)

## Task 3: no_grad, detach and frozen parameters

Return a dict with three booleans, each found by running code (not by guessing):
`'no_grad_output_requires_grad'`: does `w * 2` computed inside `torch.no_grad()` require grad (w has requires_grad=True)?
`'detach_shares_storage'`: does `w.detach()` share memory with `w`?
`'frozen_param_has_grad'`: after setting `lin.weight.requires_grad_(False)` and calling backward on `lin(x).sum()`, is `lin.weight.grad` not None?

In [ ]:
def frozen_grads():
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_frozen_grads(f):
    d = f()
    assert d == {"no_grad_output_requires_grad": False, "detach_shares_storage": True, "frozen_param_has_grad": False}, d

check("frozen_grads", _test_frozen_grads, frozen_grads)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: A custom autograd Function
class SiLUFn_solution(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * torch.sigmoid(x)
    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        s = torch.sigmoid(x)
        return grad_out * s * (1 + x * (1 - s))

check("SiLUFn (reference)", _test_SiLUFn, SiLUFn_solution)

In [ ]:
# @title Solution for Task 2: Gradient accumulation
def accumulate_solution(model, X, Y, k):
    model.zero_grad()
    for xc, yc in zip(X.chunk(k), Y.chunk(k)):
        (F.mse_loss(model(xc), yc) / k).backward()
    return {n: p.grad.clone() for n, p in model.named_parameters()}

check("accumulate (reference)", _test_accumulate, accumulate_solution)

In [ ]:
# @title Solution for Task 3: no_grad, detach and frozen parameters
def frozen_grads_solution():
    w = torch.randn(3, requires_grad=True)
    with torch.no_grad():
        a = (w * 2).requires_grad
    b = w.detach().data_ptr() == w.data_ptr()
    lin = nn.Linear(3, 2); lin.weight.requires_grad_(False); lin(torch.randn(4, 3)).sum().backward()
    return {"no_grad_output_requires_grad": a, "detach_shares_storage": b, "frozen_param_has_grad": lin.weight.grad is not None}

check("frozen_grads (reference)", _test_frozen_grads, frozen_grads_solution)

---
## Experiments
**Look at the graph.** Every result of a differentiable op has a `grad_fn` pointing at its backward node; following `next_functions` walks the graph that `.backward()` traverses.

In [ ]:
x = torch.randn(3, requires_grad=True); y = (torch.tanh(x) * 2).sum()
node, depth = y.grad_fn, 0
while node is not None and depth < 6:
    print("  " * depth + type(node).__name__); nxt = [n for n, _ in node.next_functions if n is not None]; node = nxt[0] if nxt else None; depth += 1

---
## Takeaways
* Autograd records a graph of `grad_fn` nodes during the forward pass; `.backward()` walks it in reverse and accumulates into `.grad`.
* Gradients accumulate, so call `zero_grad()` each step, and divide micro-batch losses by k to emulate a bigger batch.
* `no_grad` stops recording, `detach` returns a graph-free view, and frozen parameters get no `.grad` (the basis of LoRA-style fine-tuning).